# Home owner — PyTorch distribuido y Optuna

Clasificación binaria de `home_owner` con una MLP. `customer_id` es solamente un identificador: se excluye explícitamente de las variables de entrada y no se genera ningún embedding ni hash para él.

Cada trial de Optuna entrena su arquitectura mediante `run_with_torch_distributor` y DDP; por tanto, la creación y entrenamiento de la red se distribuyen entre los procesos disponibles.

## 0. Conexión e instalación

Ejecuta estas celdas en ADA/EMR. Ajusta el cluster y el rol si corresponden a otro entorno.

In [ ]:
%load_ext sagemaker_studio_analytics_extension.magics
%sm_analytics emr connect  \
    --cluster-id j-3PKXCEAPBCUQJ \
    --auth-type Basic_Access \
    --language python \
    --emr-execution-role-arn arn:aws:iam::430118852718:role/BBVARole-ada-us-east-1-sbx-live-s-size-dev

In [ ]:
%pip install -q optuna scikit-learn pandas awswrangler s3fs pyarrow

## 1. Librerías y sesión Spark

In [ ]:
import os
import sys
from pathlib import Path

# Soporta ejecutar el notebook desde la raíz del repo o desde notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import optuna
from dataproc_sdk.dataproc_sdk_datiopysparksession.datiopysparksession import DatioPysparkSession
from mercury.contrib.ml.nn.torch import run_with_torch_distributor

from utils.torch_tabular import (
    class_counts,
    compute_standardization_stats,
    configure_ssl,
    expand_parquet_paths,
    infer_numeric_features,
    prepare_train_validation,
    run_distributed_training,
    set_seed,
    suggest_mlp_params,
)

configure_ssl()
spark = DatioPysparkSession().get_or_create().getSparkSession()

## 2. Configuración

La lista de predictores se infiere del esquema de entrenamiento: se toman únicamente columnas numéricas y se excluyen `TARGET_COL` e `ID_COL`. Si se requieren más exclusiones de negocio, añádelas en `utils.torch_tabular.numeric_feature_columns`.

In [ ]:
TRAIN_PATH = "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myed/df_moma_comprador_casa1"
VALID_PATH = "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myed/df_moma_comprador_casa_val"

TARGET_COL = "home_owner"
ID_COL = "customer_id"  # Se excluye del modelo; no es una feature.
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")

PREPARED_BASE_PATH = os.getenv(
    "PREPARED_BASE_PATH",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myed/home_owner_torch_prepared",
)
PREPARED_TRAIN_PATH = f"{PREPARED_BASE_PATH.rstrip('/')}/train"
PREPARED_VALID_PATH = f"{PREPARED_BASE_PATH.rstrip('/')}/validation"
MODEL_ARTIFACT_PATH = os.getenv(
    "MODEL_ARTIFACT_PATH",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myed/models/home_owner_mlp_optuna.pt",
)

RUN_SPARK_PREPARE = os.getenv("RUN_SPARK_PREPARE", "true").lower() in {"1", "true", "yes"}
TRAIN_REPARTITIONS = int(os.getenv("TRAIN_REPARTITIONS", "512"))
VALID_REPARTITIONS = int(os.getenv("VALID_REPARTITIONS", "256"))

# Cada trial se distribuye con DDP en estos procesos. Optuna prueba trials secuencialmente.
NUM_PROCESSES = int(os.getenv("NUM_PROCESSES", "2"))
N_TRIALS = int(os.getenv("N_TRIALS", "20"))
TRIAL_EPOCHS = int(os.getenv("TRIAL_EPOCHS", "3"))
FINAL_EPOCHS = int(os.getenv("FINAL_EPOCHS", "10"))
SEED = int(os.getenv("SEED", "12345"))

DATA_CONFIG = {
    "batch_rows": int(os.getenv("BATCH_ROWS", "200000")),
    "batch_size": int(os.getenv("TRAIN_BATCH_SIZE", "8192")),
    "num_workers": int(os.getenv("NUM_WORKERS", "0")),
    "cache_dir": os.getenv("PARQUET_LOCAL_CACHE_DIR", "/tmp/home_owner_parquet_cache"),
    "region": AWS_REGION,
    "seed": SEED,
}

## 3. Preparación reproducible

Spark calcula las estadísticas solamente con train, aplica el z-score a ambos splits y escribe Parquet optimizado. La salida contiene exclusivamente las features numéricas y `home_owner`; `customer_id` no se escribe.

In [ ]:
set_seed(SEED)

if RUN_SPARK_PREPARE:
    FEATURE_COLS, scaler_mean, scaler_std = prepare_train_validation(
        train_path=TRAIN_PATH,
        validation_path=VALID_PATH,
        prepared_train_path=PREPARED_TRAIN_PATH,
        prepared_validation_path=PREPARED_VALID_PATH,
        target_col=TARGET_COL,
        id_col=ID_COL,
        spark=spark,
        train_repartitions=TRAIN_REPARTITIONS,
        validation_repartitions=VALID_REPARTITIONS,
    )
else:
    # Las salidas existentes deben haberse creado con la misma lista de columnas.
    FEATURE_COLS = infer_numeric_features(TRAIN_PATH, TARGET_COL, ID_COL, spark)
    scaler_mean, scaler_std = compute_standardization_stats(TRAIN_PATH, FEATURE_COLS, spark)

print(f"Features numéricas: {len(FEATURE_COLS)}")
print(FEATURE_COLS)
assert ID_COL not in FEATURE_COLS, "customer_id no puede ser variable de entrada"

TRAIN_FILES = expand_parquet_paths([PREPARED_TRAIN_PATH], region=AWS_REGION)
VALID_FILES = expand_parquet_paths([PREPARED_VALID_PATH], region=AWS_REGION)
if len(TRAIN_FILES) < NUM_PROCESSES:
    raise ValueError(
        f"Solo hay {len(TRAIN_FILES)} archivos train para {NUM_PROCESSES} procesos DDP. "
        "Reduce NUM_PROCESSES o aumenta TRAIN_REPARTITIONS."
    )

n_negative, n_positive = class_counts(PREPARED_TRAIN_PATH, TARGET_COL, spark)
if n_positive == 0 or n_negative == 0:
    raise ValueError("El target debe contener ambas clases 0 y 1 en train.")
POS_WEIGHT = n_negative / n_positive
print({"train_files": len(TRAIN_FILES), "valid_files": len(VALID_FILES), "negative": n_negative, "positive": n_positive, "pos_weight": POS_WEIGHT})

## 4. Búsqueda de arquitectura con Optuna

La métrica objetivo es F1 de validación. El espacio de búsqueda incluye cantidad y ancho de capas, dropout, activación, batch normalization, tasa de aprendizaje y regularización. Cada evaluación de `objective` lanza entrenamiento distribuido con `run_with_torch_distributor`.

In [ ]:
def objective(trial):
    model_params = suggest_mlp_params(trial)
    trial_config = {
        **DATA_CONFIG,
        "epochs": TRIAL_EPOCHS,
        "pos_weight": POS_WEIGHT,
        # No se conserva artefacto en trials; solo en el entrenamiento final.
        "artifact_path": None,
    }
    result = run_with_torch_distributor(
        train_fn=run_distributed_training,
        num_processes=NUM_PROCESSES,
        local_mode=False,
        use_gpu=False,
        train_args=(
            TRAIN_FILES,
            VALID_FILES,
            FEATURE_COLS,
            TARGET_COL,
            model_params,
            trial_config,
            None,
        ),
    )
    if result.get("status") != "ok":
        raise RuntimeError(f"Trial {trial.number} no devolvió resultado de rank 0: {result}")
    metrics = result["valid_metrics"]
    trial.set_user_attr("valid_metrics", metrics)
    return metrics["f1_score"]


sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="maximize", sampler=sampler, study_name="home_owner_mlp")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

print("Mejor F1 de validación:", study.best_value)
print("Mejores hiperparámetros:", study.best_params)
print("Métricas del mejor trial:", study.best_trial.user_attrs["valid_metrics"])

## 5. Entrenamiento final y artefacto

Se reentrena la mejor arquitectura durante más épocas y se guarda un artefacto en S3 con pesos, lista de features y estadísticos de normalización. Así la inferencia puede reproducir exactamente las transformaciones, sin requerir `customer_id`.

In [ ]:
best_model_params = suggest_mlp_params(optuna.trial.FixedTrial(study.best_params))
final_config = {
    **DATA_CONFIG,
    "epochs": FINAL_EPOCHS,
    "pos_weight": POS_WEIGHT,
    "artifact_path": MODEL_ARTIFACT_PATH,
}
artifact_metadata = {
    "scaler_mean": scaler_mean.tolist(),
    "scaler_std": scaler_std.tolist(),
    "id_col_excluded": ID_COL,
    "best_trial": study.best_trial.number,
    "best_trial_f1_score": study.best_value,
}

final_result = run_with_torch_distributor(
    train_fn=run_distributed_training,
    num_processes=NUM_PROCESSES,
    local_mode=False,
    use_gpu=False,
    train_args=(
        TRAIN_FILES,
        VALID_FILES,
        FEATURE_COLS,
        TARGET_COL,
        best_model_params,
        final_config,
        artifact_metadata,
    ),
)

if final_result.get("status") != "ok":
    raise RuntimeError(f"El entrenamiento final no devolvió resultado de rank 0: {final_result}")

print("Métricas finales:", final_result["valid_metrics"])
print("Artefacto:", final_result["artifact_path"])

## Notas operativas

- `run_with_torch_distributor` distribuye los procesos de cada trial; no se ejecutan varios trials simultáneamente para no competir por los mismos recursos del clúster.
- El dataset descarga el shard de cada rank a `/tmp` mediante boto3 y PyArrow lo lee localmente. Esto evita los errores SSL de `pyarrow.fs.S3FileSystem` observados dentro de workers DDP.
- Para repetir sin reescribir los Parquet estandarizados, usa `RUN_SPARK_PREPARE=false`. Conserva las mismas rutas y el mismo esquema.